# Document QA System (RAG)

A Retrieval-Augmented Generation system that answers questions about your documents.

**Use cases:** Company handbook QA, research paper assistant, course materials search.

### Architecture

```
Upload Documents
      ↓
Text Extraction (PyPDF / TextLoader)
      ↓
Chunking (RecursiveCharacterTextSplitter)
      ↓
Embedding API (HuggingFace Inference API)
      ↓
Vector Store (InMemoryVectorStore)
      ↓
User Question → Embed → Vector Search → Top-K Chunks
      ↓
LLM API via HuggingFace
      ↓
Answer with Source Citations
```

**No local LLM hosting required** — all inference runs via the free HuggingFace Inference API.

In [2]:
import os
import warnings
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from huggingface_hub import InferenceClient


warnings.filterwarnings("ignore", category=DeprecationWarning)

/opt/homebrew/Caskroom/miniconda/base/envs/venv_nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Configuration

Get a free HuggingFace API token at https://huggingface.co/settings/tokens

In [ ]:
# --- API Token ---
HF_TOKEN = os.environ.get("HUGGINGFACE_API_KEY", None)

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Enter your HuggingFace API token: ")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

# --- Models ---
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL")
LLM_MODEL = os.environ.get("LLM_MODEL")
# Alternatives if the above is unavailable:
# LLM_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
# LLM_MODEL = "microsoft/Phi-3.5-mini-instruct"

# --- Chunking ---
CHUNK_SIZE = int(os.environ.get("CHUNK_SIZE"))
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP"))

# --- Retrieval ---
NUM_RETRIEVED_DOCS = int(os.environ.get("NUM_RETRIEVED_DOCS"))

In [4]:
DOCS_DIR = Path("./uploaded_docs")
DOCS_DIR.mkdir(exist_ok=True)


def upload_documents():
    """Upload documents via Colab widget or list files in the docs directory."""
    print(f"Place your PDF/TXT/MD files in: {DOCS_DIR.resolve()}")
    print("Then re-run this cell.")

    files = [f for f in DOCS_DIR.iterdir() if f.is_file()]
    print(f"\nFound {len(files)} file(s): {[f.name for f in files]}")
    return files


def load_documents(file_paths):
    """Load documents from file paths. Returns a list of langchain Documents."""
    all_docs = []

    for path in file_paths:
        path = Path(path)
        try:
            if path.suffix.lower() == ".pdf":
                loader = PyPDFLoader(str(path))
                docs = loader.load()
                print(f"  PDF:  {path.name} ({len(docs)} pages)")
            elif path.suffix.lower() in (".txt", ".md"):
                loader = TextLoader(str(path), encoding="utf-8")
                docs = loader.load()
                print(f"  Text: {path.name}")
            else:
                print(f"  Skipping unsupported file: {path.name}")
                continue
            all_docs.extend(docs)
        except Exception as e:
            print(f"  Error loading {path.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs


file_paths = upload_documents()
documents = load_documents(file_paths)

Place your PDF/TXT/MD files in: /Users/rafi/Code/python/rag_live/notebooks/uploaded_docs
Then re-run this cell.

Found 1 file(s): ['1984.pdf']
  PDF:  1984.pdf (182 pages)

Total documents loaded: 182


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(documents)

print(f"Documents: {len(documents)} → Chunks: {len(chunks)}")
print(f"Chunk size: {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP}")

if chunks:
    print(f"\n--- First chunk preview ---")
    print(f"Content ({len(chunks[0].page_content)} chars):")
    print(chunks[0].page_content[:300] + "...")
    print(f"Metadata: {chunks[0].metadata}")

Documents: 182 → Chunks: 182
Chunk size: 5000 chars, overlap: 200

--- First chunk preview ---
Content (23 chars):
1984
George Orwell
1949...
Metadata: {'producer': 'pdfTeX-1.0a-pdfcrypt', 'creator': 'PyPDF', 'creationdate': 'D:20020317181800', 'source': 'uploaded_docs/1984.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}


In [6]:
embeddings = HuggingFaceEndpointEmbeddings(
    model=EMBEDDING_MODEL,
    huggingfacehub_api_token=HF_TOKEN,
)

# Verify the embedding model works
test_embedding = embeddings.embed_query("test")
print(f"Embedding dim:   {len(test_embedding)}")

# Build in-memory vector store from all chunks
print(f"\nEmbedding and indexing {len(chunks)} chunks...")
vectorstore = InMemoryVectorStore.from_documents(chunks, embedding=embeddings)
print("InMemoryVectorStore ready")

Embedding dim:   384

Embedding and indexing 182 chunks...
InMemoryVectorStore ready


In [7]:
client = InferenceClient(model=LLM_MODEL, token=HF_TOKEN)

# Quick test
print(f"LLM: {LLM_MODEL}")
test_response = client.chat_completion(
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=64,
)
print(f"LLM test: {test_response.choices[0].message.content}")

# Prompt template for RAG
QA_PROMPT = PromptTemplate.from_template(
    """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Question: {question}

Helpful Answer:"""
)

print("\nRetriever + LLM ready!")

LLM: Qwen/Qwen3-4B-Instruct-2507
LLM test: Hello! 😊

Retriever + LLM ready!


In [8]:
def ask_question(question, show_sources=True):
    """Ask a question about the loaded documents."""
    required_objects = ["vectorstore", "client", "QA_PROMPT"]
    missing = [name for name in required_objects if name not in globals()]
    if missing:
        raise RuntimeError(
            f"Missing setup objects: {missing}. Run sections 2-7 (or load index in section 10) first."
        )

    print(f"Q: {question}")
    print("-" * 60)

    docs = vectorstore.similarity_search(question, k=NUM_RETRIEVED_DOCS)

    # Build prompt with context
    context = "\n\n".join(doc.page_content for doc in docs)
    prompt = QA_PROMPT.format(context=context, question=question)

    # Generate answer via chat completion
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.7,
    )
    answer = response.choices[0].message.content.strip()
    print(f"\nA: {answer}")

    if show_sources and docs:
        print(f"\n{'=' * 60}")
        print(f"Sources ({len(docs)} chunks):")
        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get("source", "unknown")
            page = doc.metadata.get("page", "N/A")
            print(f"\n  [{i}] {Path(source).name}, page {page}")
            print(f"      {doc.page_content[:150]}...")

    return answer

In [9]:
answer = ask_question("What is the main topic of the documents?")
answer

Q: What is the main topic of the documents?
------------------------------------------------------------

A: The main topic of the documents is the manipulation and control of history and information by the Party in Oceania, specifically through the rewriting and falsification of past records to align with the ideology of Ingsoc (English Socialism). This includes the transformation of historical writings into ideologically aligned versions, the destruction of original documents, and the creation of false or altered records to serve Party propaganda. The documents highlight the process of forgery, the use of Newspeak to reshape language and thought, and the systematic erasure of inconvenient or contradictory historical facts.

Sources (3 chunks):

  [1] 1984.pdf, page 24
      actors specially chosen for their skill in imitating voices. There were the armies
of reference clerks whose job was simply to draw up lists of books ...

  [2] 1984.pdf, page 181
      It would have been quite im

'The main topic of the documents is the manipulation and control of history and information by the Party in Oceania, specifically through the rewriting and falsification of past records to align with the ideology of Ingsoc (English Socialism). This includes the transformation of historical writings into ideologically aligned versions, the destruction of original documents, and the creation of false or altered records to serve Party propaganda. The documents highlight the process of forgery, the use of Newspeak to reshape language and thought, and the systematic erasure of inconvenient or contradictory historical facts.'

In [10]:
# Debug: isolate which step crashes the kernel
import sys

print("Step 1: Testing embedding API...")
sys.stdout.flush()
test_vec = embeddings.embed_query("test query")
print(f"  OK — got {len(test_vec)}-dim vector")

print("Step 2: Testing vectorstore search...")
sys.stdout.flush()
docs = vectorstore.similarity_search("test query", k=2)
print(f"  OK — got {len(docs)} docs")

print("Step 3: Testing LLM call...")
sys.stdout.flush()
resp = client.chat_completion(
    messages=[{"role": "user", "content": "Say hi"}],
    max_tokens=32,
    temperature=0,
)
print(f"  OK — {resp.choices[0].message.content}")

print("Step 4: Full ask_question...")
sys.stdout.flush()
ask_question("What is this document about?")

Step 1: Testing embedding API...
  OK — got 384-dim vector
Step 2: Testing vectorstore search...
  OK — got 2 docs
Step 3: Testing LLM call...
  OK — Hello! 🌟 How can I assist you today? 😊
Step 4: Full ask_question...
Q: What is this document about?
------------------------------------------------------------

A: The document is about rewriting a news article from *The Times* that criticizes Big Brother's Order for the Day. Specifically, the article is described as "extremely unsatisfactory" and contains references to non-existent persons ("unpersons"). The directive instructs the editor to rewrite the article in full, ensuring it aligns with Party doctrine, and to submit the revised draft to higher authority before filing it. The original article praised an organization (FFCC) that supplies comforts to sailors and highlighted a Comrade Withers, a prominent Inner Party member, who was awarded advance copies of a new publication. The task involves correcting and reworking the content to

'The document is about rewriting a news article from *The Times* that criticizes Big Brother\'s Order for the Day. Specifically, the article is described as "extremely unsatisfactory" and contains references to non-existent persons ("unpersons"). The directive instructs the editor to rewrite the article in full, ensuring it aligns with Party doctrine, and to submit the revised draft to higher authority before filing it. The original article praised an organization (FFCC) that supplies comforts to sailors and highlighted a Comrade Withers, a prominent Inner Party member, who was awarded advance copies of a new publication. The task involves correcting and reworking the content to reflect the Party\'s desired narrative.'

In [11]:
print("Document QA — type your questions (type 'quit' to stop)\n")

while True:
    question = input("\nYour question: ").strip()
    if not question:
        continue
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    try:
        ask_question(question)
    except Exception as e:
        print(f"Error: {e}")

Document QA — type your questions (type 'quit' to stop)

Q: What is this about?
------------------------------------------------------------

A: This is about a dystopian society in George Orwell's novel *1984*, where the Party controls information, language, and reality through propaganda, surveillance, and the suppression of truth. Key elements include the manipulation of language via Newspeak, the persecution of dissenters (such as thought-criminals executed in the Ministry of Love), the manipulation of history and reality through propaganda (e.g., the false statistics shown on telescreens), and the oppressive regime's control over individuals' thoughts and emotions. The narrative also explores themes of freedom, love, and the struggle for individuality in a world where truth is systematically erased.

Sources (3 chunks):

  [1] 1984.pdf, page 28
      would talk with a disagreeable gloating satisfaction of helicopter raids on enemy
villages, and trials and confessions of thought-cr

In [12]:
# # Save
# INDEX_PATH = "./inmemory_vectorstore.json"
# vectorstore.dump(INDEX_PATH)
# print(f"Vectorstore saved to {INDEX_PATH}")

In [13]:
# # Load (run this instead of sections 4-6 to skip re-embedding)
# required = ["EMBEDDING_MODEL", "HF_TOKEN"]
# missing = [name for name in required if name not in globals()]
# if missing:
#     raise RuntimeError(f"Missing config values: {missing}. Run section 3 first.")

# INDEX_PATH = "./inmemory_vectorstore.json"
# embeddings = HuggingFaceEndpointEmbeddings(
#     model=EMBEDDING_MODEL,
#     huggingfacehub_api_token=HF_TOKEN,
# )

# vectorstore = InMemoryVectorStore.load(INDEX_PATH, embedding=embeddings)
# print("Vectorstore loaded — ready for questions!")